In [1]:
from pathlib import Path

import polars as pl
import scipy as sp

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dqdvs import dqdv_histogram, dqdv_finite_differences

from config import DATA_PATH
ZENODO_DATA_PATH = Path(DATA_PATH) / Path("HALF_CELL_OCVS_ZENODO")

In [2]:
################################ LOAD DATA  ###################################
all_filepaths = ZENODO_DATA_PATH.rglob("*.parquet")

## IC methods

In [3]:
sg_window_sizes = [20, 100, 500] #ca 1 mV, 5 mV, 10 mV
h_bin_sizes = [1e-3, 5e-3, 10e-3]

selected_materials = {
    'sintef__sintef-lfp-R2032-gelon-fbeeaf__20250607__p-ocvhold__RT.bdf.parquet':"LFP",
    'sintef__sintef-nmc111-R2032-customcells-1e88d8__20250625__p-ocvhold__RT.bdf.parquet':"NMC111",
    'sintef__sintef-graphite-R2032-intelligent-677295__20250514__p-ocvhold__RT.bdf.parquet':"Graphite",
}

In [4]:
colors = px.colors.qualitative.Plotly

In [18]:
fig = make_subplots(cols=1, rows=len(selected_materials), shared_xaxes=True, vertical_spacing=0.02)

fig.update_layout(
    template="simple_white",
    margin=dict(l=10, r=10, t=60, b=20),
    width=400, 
    height=500,
    legend=dict(
        orientation="h",    # horizontal
        yanchor="bottom",   # anchor legend's bottom
        y=0.98,             # place above the plot area
        xanchor="center",
        x=0.5
    ),)

for mn, (material, material_label) in enumerate(selected_materials.items(), start=1):

    df = pl.read_parquet(ZENODO_DATA_PATH / Path(material))

    df_subset = (df
                .filter(pl.col('Step Index / 1')==2)
                .filter(pl.col('Cycle Count / 1')==3)
                )

    q = df_subset['Cumulative Capacity / Ah'].to_numpy()
    v = df_subset['Voltage / V'].to_numpy()

    q = q/max(q)

    fig.add_trace(go.Scatter(x=q, 
                         y=v, 
                         name="Original", 
                         mode="lines",
                         showlegend=True if mn==1 else False,
                         line=dict(color="Black", width=5)), row=mn, col=1)

    
    for wn, window_size in enumerate(sg_window_sizes, start=0):

        v_smooth = sp.signal.savgol_filter(v, window_length=window_size, polyorder=1)
        v_dqdv, dqdv = dqdv_finite_differences(q, v_smooth)
        q_rec = sp.integrate.cumulative_trapezoid(dqdv, v_dqdv, initial=0.0)

        fig.add_trace(go.Scatter(x=q_rec, 
                            y=v_dqdv, 
                            name=f"SG-FD {window_size} pts.", 
                            mode="lines",
                            showlegend=True if mn==1 else False,
                            line=dict(color=colors[wn], width=2)), row=mn, col=1)


fig.add_annotation(x=0.05, 
                y=3.3,
                text="LFP",
                showarrow=False,
                font=dict(size=15, weight=800), 
                xanchor="left",
                row=1, col=1)

fig.add_annotation(x=0.05, 
                y=3.85,
                text="NMC111",
                showarrow=False,
                font=dict(size=15, weight=800), 
                xanchor="left",
                row=2, col=1)

fig.add_annotation(x=0.05, 
                y=0.5,
                text="Graphite",
                showarrow=False,
                font=dict(size=15, weight=800), 
                xanchor="left",
                row=3, col=1)

fig.update_xaxes(showgrid=False)
fig.update_xaxes(title_text="Normalized Cumulative Capacity / 1", col=1, row=3)

fig.update_layout(title=dict(text="<b>a</b>. Reconstruction: Finite Differences", 
                            yanchor="bottom",   # anchor legend's bottom
                            y=0.97,             # place above the plot area
                            xanchor="left",
                            x=0.05))

fig.update_yaxes(title_text="Voltage / V", showgrid=False)
fig.update_xaxes(range=[0, 1.0])
fig.show()

    

c:\Users\eibarc\Documents\Repositories\incremental-capacity-curves\dqdvs.py:27: RuntimeWarning:

divide by zero encountered in divide



In [19]:
fig.write_image("figures/reconstruction_finite_differences.png", scale=5)

In [20]:
fig = make_subplots(cols=1, rows=len(selected_materials), shared_xaxes=True, vertical_spacing=0.02)

fig.update_layout(
    template="simple_white",
    margin=dict(l=10, r=10, t=60, b=20),
    width=400, 
    height=500,
    legend=dict(
        orientation="h",    # horizontal
        yanchor="bottom",   # anchor legend's bottom
        y=0.98,             # place above the plot area
        xanchor="center",
        x=0.5
    ),)

for mn, material in enumerate(selected_materials, start=1):

    df = pl.read_parquet(ZENODO_DATA_PATH / Path(material))

    df_subset = (df
                .filter(pl.col('Step Index / 1')==2)
                .filter(pl.col('Cycle Count / 1')==3)
                )

    q = df_subset['Cumulative Capacity / Ah'].to_numpy()
    v = df_subset['Voltage / V'].to_numpy()
    
    q = q/max(q)

    fig.add_trace(go.Scatter(x=q, 
                         y=v, 
                         name="Original", 
                         mode="lines",
                         showlegend=True if mn==1 else False,
                         line=dict(color="Black", width=5)), row=mn, col=1)
    
    for wn, bin_size in enumerate(h_bin_sizes, start=0):

        v_h, dqdv_h = dqdv_histogram(q, v, bin_size)
        q_rec = sp.integrate.cumulative_trapezoid(dqdv_h, v_h, initial=0.0)

        fig.add_trace(go.Scatter(x=q_rec, 
                            y=v_h, 
                            name=f"Bin: {bin_size*1e3} mV", 
                            mode="lines",
                            showlegend=True if mn==1 else False,
                            line=dict(color=colors[wn], width=2)), row=mn, col=1)


fig.add_annotation(x=0.05, 
                y=3.3,
                text="LFP",
                showarrow=False,
                font=dict(size=15, weight=800), 
                xanchor="left",
                row=1, col=1)

fig.add_annotation(x=0.05, 
                y=3.85,
                text="NMC111",
                showarrow=False,
                font=dict(size=15, weight=800), 
                xanchor="left",
                row=2, col=1)

fig.add_annotation(x=0.05, 
                y=0.5,
                text="Graphite",
                showarrow=False,
                font=dict(size=15, weight=800), 
                xanchor="left",
                row=3, col=1)


fig.update_layout(title=dict(text="<b>b</b>. Reconstruction: Histograms", 
                            yanchor="bottom",   # anchor legend's bottom
                            y=0.97,             # place above the plot area
                            xanchor="left",
                            x=0.05)
        )

fig.update_xaxes(showgrid=False)
fig.update_xaxes(title_text="Normalized Cumulative Capacity / 1", col=1, row=3)

fig.update_yaxes(showgrid=False)
fig.update_yaxes(title_text="Voltage / V")
fig.update_xaxes(range=[0, 1.0])
fig.show()

    

In [21]:
fig.write_image("figures/reconstruction_histograms.png", scale=5)